<a href="https://colab.research.google.com/github/LinarBiktimerov/python-ai-Biktimerov-Linar/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных о автопроизводителях

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`Automaker-companies-date-workers-profit.csv`](https://github.com/LinarBiktimerov/python-ai-Biktimerov-Linar/blob/main/data/Automaker-companies-date-workers-profit.csv) — информация об автопроизводителях: компания, страна, год основания, количество сотрудников и выручка

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем CSV-файл в pandas DataFrame
3. Очищаем и переименовываем столбцы при необходимости
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [1]:
# 🐱 Шаг 1. Клонируем ваш репозиторий в Colab

import os

# 🔧 ИЗМЕНЕНО: имя репозитория на ваше
repo = "python-ai-Biktimerov-Linar"
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Biktimerov-Linar
    # 🔧 ИЗМЕНЕНО: URL репозитория на ваш
    !git clone -q https://github.com/LinarBiktimerov/python-ai-Biktimerov-Linar.git

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

/content/python-ai-Biktimerov-Linar
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Biktimerov-Linar


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [2]:
# 🐱 Шаг 2A. Чтение CSV-файла в pandas

import pandas as pd

# 🔧 ИЗМЕНЕНО: читаем ваш файл с данными об автопроизводителях
df_automakers = pd.read_csv("data/Automaker-companies-date-workers-profit.csv")

print("✅ Загружено строк в df_automakers:", len(df_automakers))
print("✅ Столбцы:", df_automakers.columns.tolist())

✅ Загружено строк в df_automakers: 91
✅ Столбцы: ['company', 'companyLabel', 'countryLabel', 'founded', 'employees', 'revenue']


## 🧹 [2B] Очистка и переименование столбцов

В вашем CSV-файле есть **технические столбцы** из Wikidata, которые мешают простому анализу:

- Столбец `company` с URL (ссылкой на объект Wikidata, например `http://www.wikidata.org/entity/Q9584`) — **удаляем**, так как для базового анализа он не нужен.
- Столбцы `companyLabel`, `countryLabel` содержат читаемые названия — **переименуем**, убрав постфикс `Label`.

В этом шаге мы:
- удалим столбец с URI Wikidata (`company`);
- переименуем `companyLabel → company`, `countryLabel → country`;
- приведём числовые столбцы (`employees`, `revenue`) к типу `int64`/`float64`;
- при необходимости обработаем столбец `founded` как дату.

При приведении к числам мы используем:
- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец с URI можно сохранить, если позже понадобится переходить к оригинальным записям в Викиданных.

In [8]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для df_automakers

# 🔧 Проверка: есть ли столбцы с постфиксом Label (нужна ли очистка?)
if "companyLabel" in df_automakers.columns:

    # 1) Удаляем технический столбец с URI Wikidata
    if "company" in df_automakers.columns:
        df_automakers = df_automakers.drop(columns=["company"])
        print("🗑️ Удалён столбец 'company' (URI Wikidata)")

    # 2) Переименовываем столбцы: убираем постфикс Label
    df_automakers = df_automakers.rename(columns={
        "companyLabel": "company",      # ← Honda, Nissan, Mazda
        "countryLabel": "country",      # ← Япония, США, Германия
    })
    print("✏️ Переименованы: companyLabel → company, countryLabel → country")

    # 3) Приводим числовые столбцы к правильному типу
    # 🔧 Внимание: revenue может быть очень большим числом — используем float для безопасности
    df_automakers["employees"] = pd.to_numeric(
        df_automakers["employees"], errors="coerce"
    ).fillna(0).astype("Int64")  # Int64 поддерживает пропуски

    df_automakers["revenue"] = pd.to_numeric(
        df_automakers["revenue"], errors="coerce"
    ).fillna(0)  # оставляем как float, т.к. выручка может быть огромной

    # 4) Опционально: приводим founded к типу datetime
    df_automakers["founded"] = pd.to_datetime(
        df_automakers["founded"], errors="coerce"
    )

    print("✅ df_automakers очищен и готов к анализу")

else:
    print("⏭️ df_automakers уже очищен, пропускаем")

# 🔍 Быстрая проверка результата
print("\n📋 Столбцы после очистки:", df_automakers.columns.tolist())
display(df_automakers.head())

⏭️ df_automakers уже очищен, пропускаем

📋 Столбцы после очистки: ['company', 'country', 'founded', 'employees', 'revenue']


,company,country,founded,employees,revenue
0,Honda,Япония,1948-09-24 00:00:00+00:00,204035,1.495000e+13
1,Honda,Япония,1948-09-24 00:00:00+00:00,204035,1.495000e+13
2,Nissan,Япония,1933-12-26 00:00:00+00:00,136134,1.195117e+13
3,Mazda,Япония,1920-01-30 00:00:00+00:00,48750,3.826752e+12
4,Yamaha Motor Company,Япония,1955-07-01 00:00:00+00:00,53701,2.414759e+12


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame `df_automakers`:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- проверим типы данных и наличие пропусков (`info()`);
- посчитаем базовую статистику по числовым столбцам (`employees`, `revenue`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы быстро выводить основную информацию о датасете. Эта функция универсальна и пригодится в будущих анализах.

> 💡 **Совет:** Всегда начинайте анализ с быстрого обзора данных — это помогает сразу заметить аномалии, пропуски или неверные типы данных.

In [9]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов, типы и первые строки."""
    print(f"\n📊 {name}")
    print("═" * 50)
    print("📏 Размер (строки, столбцы):", df.shape)
    print("📋 Столбцы:", ", ".join(df.columns))
    print("\n🔍 Типы данных и пропуски:")
    print(df.info())
    print(f"\n📄 Первые {n} строк:")
    display(df.head(n))
    print("\n📈 Базовая статистика (числовые столбцы):")
    display(df.describe(include='number'))

# 🔍 Шаг 3. Обзор данных для автопроизводителей

show_info(df_automakers, "Автопроизводители: компании, сотрудники, выручка (df_automakers)")

# 🎁 Бонус: быстрый анализ по ключевым метрикам
print("\n🚀 Топ-5 компаний по количеству сотрудников:")
display(df_automakers.nlargest(5, "employees")[["company", "country", "employees"]])

print("\n💰 Топ-5 компаний по выручке:")
display(df_automakers.nlargest(5, "revenue")[["company", "country", "revenue"]])


📊 Автопроизводители: компании, сотрудники, выручка (df_automakers)
══════════════════════════════════════════════════
📏 Размер (строки, столбцы): (91, 5)
📋 Столбцы: company, country, founded, employees, revenue

🔍 Типы данных и пропуски:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype              
---  ------     --------------  -----              
 0   company    91 non-null     object             
 1   country    91 non-null     object             
 2   founded    91 non-null     datetime64[ns, UTC]
 3   employees  91 non-null     Int64              
 4   revenue    91 non-null     float64            
dtypes: Int64(1), datetime64[ns, UTC](1), float64(1), object(2)
memory usage: 3.8+ KB
None

📄 Первые 5 строк:


,company,country,founded,employees,revenue
0,Honda,Япония,1948-09-24 00:00:00+00:00,204035,1.495000e+13
1,Honda,Япония,1948-09-24 00:00:00+00:00,204035,1.495000e+13
2,Nissan,Япония,1933-12-26 00:00:00+00:00,136134,1.195117e+13
3,Mazda,Япония,1920-01-30 00:00:00+00:00,48750,3.826752e+12
4,Yamaha Motor Company,Япония,1955-07-01 00:00:00+00:00,53701,2.414759e+12



📈 Базовая статистика (числовые столбцы):


,employees,revenue
count,91.0,9.100000e+01
mean,57271.32967,6.041601e+11
std,98107.669908,2.537224e+12
min,18.0,0.000000e+00
25%,3628.5,9.899530e+08
50%,13550.0,1.038008e+10
75%,69683.0,1.282867e+11
max,672800.0,1.495000e+13



🚀 Топ-5 компаний по количеству сотрудников:


,company,country,employees
15,Volkswagen,Германия,672800
10,Toyota,Япония,359542
16,Stellantis,Нидерланды,258275
0,Honda,Япония,204035
1,Honda,Япония,204035



💰 Топ-5 компаний по выручке:


,company,country,revenue
0,Honda,Япония,1.495000e+13
1,Honda,Япония,1.495000e+13
2,Nissan,Япония,1.195117e+13
3,Mazda,Япония,3.826752e+12
4,Yamaha Motor Company,Япония,2.414759e+12


## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** компаний и стран есть в данных;
- **какие страны встречаются чаще всего** (Топ‑5 по числу компаний);
- **диапазон годов основания** компаний (минимальный и максимальный год);
- **базовую статистику** по количеству сотрудников и выручке.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_automakers["country"].value_counts().head()` даёт **Топ‑5 стран по числу записей**.

> 💡 **Зачем это нужно:** Быстрая валидация помогает обнаружить аномалии — например, компании с нулевой выручкой, отрицательным числом сотрудников или некорректными датами основания.


In [10]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# 📊 Датасет: Автопроизводители
print("\n📊 Датасет: Компании, сотрудники и выручка")
print("Уникальных компаний в df_automakers:", df_automakers["company"].nunique())
print("Уникальных стран:", df_automakers["country"].nunique())

print("\n🌍 Топ-5 стран по числу компаний:")
print(df_automakers["country"].value_counts().head())

print("\n📅 Диапазон годов основания:")
print("Минимальный год:", df_automakers["founded"].dt.year.min())
print("Максимальный год:", df_automakers["founded"].dt.year.max())

print("\n👥 Статистика по количеству сотрудников:")
print(df_automakers["employees"].describe())

print("\n💰 Статистика по выручке:")
print(df_automakers["revenue"].describe())

# 🔎 Проверка на аномалии
print("\n⚠️ Проверка на возможные аномалии:")
print("Компаний с 0 сотрудников:", (df_automakers["employees"] == 0).sum())
print("Компаний с 0 выручкой:", (df_automakers["revenue"] == 0).sum())
print("Компаний с пропусками в employees:", df_automakers["employees"].isna().sum())
print("Компаний с пропусками в revenue:", df_automakers["revenue"].isna().sum())

print("\n✅ Валидация завершена")

🔍 Быстрая проверка данных

📊 Датасет: Компании, сотрудники и выручка
Уникальных компаний в df_automakers: 65
Уникальных стран: 24

🌍 Топ-5 стран по числу компаний:
country
Германия    12
Китай        9
Чехия        9
Беларусь     9
Япония       8
Name: count, dtype: int64

📅 Диапазон годов основания:
Минимальный год: 1857
Максимальный год: 2022

👥 Статистика по количеству сотрудников:
count            91.0
mean      57271.32967
std      98107.669908
min              18.0
25%            3628.5
50%           13550.0
75%           69683.0
max          672800.0
Name: employees, dtype: Float64

💰 Статистика по выручке:
count    9.100000e+01
mean     6.041601e+11
std      2.537224e+12
min      0.000000e+00
25%      9.899530e+08
50%      1.038008e+10
75%      1.282867e+11
max      1.495000e+13
Name: revenue, dtype: float64

⚠️ Проверка на возможные аномалии:
Компаний с 0 сотрудников: 0
Компаний с 0 выручкой: 1
Компаний с пропусками в employees: 0
Компаний с пропусками в revenue: 0

✅ Валидаци

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨